## Job Description Skill Extractor
 **Problem statement**

Job descriptions often contain important hiring requirements hidden within long paragraphs. Extracting structured information manually is inefficient. The objective is to build a system that identifies required skills, experience, and education directly from the input text without generating or assuming missing details.


In [1]:
pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.2 MB/s eta 0:00:00


In [2]:
pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
import os
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI #Lanchain Google GeminiAI
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List,Optional


os.environ['GOOGLE_API_KEY'] = userdata.get('api_key')

In [6]:
model_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",
                                      temperature = 0)

In [7]:
# Step1 : User Data

Job_description = """Summary:As a Data Science Practitioner, you will be engaged in formulating, designing, and delivering AI and machine learning-based decision-making frameworks and models that drive business outcomes. Your typical day will involve collaborating with various teams to measure and justify the value of AI and machine learning solutions, ensuring that they align with organizational goals and deliver tangible results. You will also be responsible for analyzing data, interpreting results, and providing insights that inform strategic decisions, all while fostering a culture of innovation and continuous improvement within the team.

Roles & Responsibilities:
Expected to be an SME.
Collaborate and manage the team to perform.
Responsible for team decisions.
Engage with multiple teams and contribute on key decisions.
Provide solutions to problems for their immediate team and across multiple teams.
Facilitate knowledge sharing and mentorship within the team to enhance overall capabilities.
Develop and implement best practices for data science methodologies and processes.

Professional & Technical

Skills:

Must To Have

Skills:

Proficiency in Data Science.
Strong analytical skills with the ability to interpret complex data sets.
Experience with machine learning frameworks and libraries such as TensorFlow or PyTorch.
Proficiency in programming languages such as Python or R for data analysis.
Familiarity with data visualization tools to effectively communicate insights.

Additional Information:
The candidate should have minimum 7.5 years of experience in Data Science.
This position is based at our Hyderabad office.
A 15 years full time education is required.

 Qualification 15 years full time education
Role: Data Science & Machine Learning - Other
Industry Type: IT Services & Consulting
Department: Data Science & Analytics
Employment Type: Full Time, Permanent
Role Category: Data Science & Machine Learning
Education
UG: B.Tech / B.E. in Any Specialization
PG: Any Postgraduate
Key Skills
Skills highlighted with ‘‘ are preferred keyskills
data science
pythondata analysismachine learning frameworksaivisualization toolsmachine learningprogramming languagesanalytical skillsrtensorflowdesignpytorchdata visualizationprogrammingml """



In [8]:
print(Job_description)

Summary:As a Data Science Practitioner, you will be engaged in formulating, designing, and delivering AI and machine learning-based decision-making frameworks and models that drive business outcomes. Your typical day will involve collaborating with various teams to measure and justify the value of AI and machine learning solutions, ensuring that they align with organizational goals and deliver tangible results. You will also be responsible for analyzing data, interpreting results, and providing insights that inform strategic decisions, all while fostering a culture of innovation and continuous improvement within the team.

Roles & Responsibilities:
Expected to be an SME.
Collaborate and manage the team to perform.
Responsible for team decisions.
Engage with multiple teams and contribute on key decisions.
Provide solutions to problems for their immediate team and across multiple teams.
Facilitate knowledge sharing and mentorship within the team to enhance overall capabilities.
Develop a

In [9]:
# step2 - Defining a Schema using the Pydantic modules

class JobDescription(BaseModel):
  Skills: list[str] = Field(description="The Skills Required for this Job")
  Experience: str = Field(description="The Experience Required")
  Education: str = Field(description="The Over all Education Qualification")

In [10]:
parser = JsonOutputParser(pydantic_object=JobDescription)

In [11]:
# Step 3 : Prompt Details
# format instructions are based on the pydantic module and json parser selected

prompt = ChatPromptTemplate.from_template(
"""
You are an Expert Job Description Data Extractor. Ensure that all relevant information is extracted from the provided Job Description in given format.
{format_instructions}

Job Description:
{Job_Description}
"""
).partial(format_instructions=parser.get_format_instructions())

In [12]:
# step 4: chain all together and invoke
# Case1:
chain = prompt | model_gemini | parser
chain.invoke({"Job_Description":Job_description})

{'Skills': ['Data Science',
  'analytical skills',
  'machine learning frameworks',
  'TensorFlow',
  'PyTorch',
  'Python',
  'R',
  'data visualization tools'],
 'Experience': 'minimum 7.5 years of experience in Data Science',
 'Education': '15 years full time education'}

In [13]:
# Case2:
Jd = "Looking for Python developer with 3 years experience"

In [14]:
chain.invoke({"Job_Description":Jd})

{'Skills': ['Python'], 'Experience': '3 years', 'Education': 'Not Specified'}

In [15]:
# Case 3:
Jd = "Looking for Oracle Technical Consultant with 4+ Years Experience, must have skills RICEW, SQL, PLSQL, Java, RESTAPI"

In [16]:
chain.invoke({"Job_Description":Jd})

{'Skills': ['RICEW', 'SQL', 'PLSQL', 'Java', 'RESTAPI'],
 'Experience': '4+ Years',
 'Education': 'Oracle Technical Consultant'}